# DINOv2 ViT-S/14: train on Google Colab

A Vision Transformer with DINOv2 self-supervised weights. See `baseline/MultiTaskBaseline/DINOv2/README.md`. The first run downloads the pretrained weights (~85 MB) from Hugging Face. A warning about a missing `HF_TOKEN` is safe to ignore.

This notebook runs the project's own scripts (`baseline/MultiTaskBaseline/DINOv2/train.py` and `evaluate.py`), so it
always uses the latest code from GitHub. It trains three models and evaluates them:

| Run | Output folder |
|---|---|
| Multi-task (one model, crop head + disease head) | `outputs/multitask_dinov2` |
| Independent crop model | `outputs/indep_crop_dinov2` |
| Independent disease model | `outputs/indep_disease_dinov2` |
| Evaluation | `outputs/multitask_dinov2/eval`, `outputs/indep_dinov2/eval` |

Expect about 50 minutes per run on a T4 GPU (3 runs in total).

## Before the first run (once per person)

1. **GPU:** Runtime → Change runtime type → **T4 GPU**.
2. **Team folder:** the data lives in the team's shared Drive folder `AML_Group_Project/Dataset`
   (the 3 CSVs + `DPD_Small.7z`). Colab can't see *Shared with me*, so add a shortcut once:
   Google Drive → **Shared with me** → right-click `AML_Group_Project` → **Organize → Add shortcut**
   → **My Drive**. Colab then finds it at `/content/drive/MyDrive/AML_Group_Project`.
3. **Results** are saved to `AML_Group_Project/aml_runs`, so the whole team sees them. This needs
   *Editor* access to the folder. With *Viewer* access, set `OUT_DIR` below to a folder in your
   own Drive.

## If Colab disconnects

Results are saved straight to Drive. Reconnect and choose **Runtime → Run all**: finished runs are
skipped. A run that was cut off restarts from epoch 1, because `train.py` can't resume yet.

Results go to one shared folder, so agree on who trains what. Two people training the same run
at the same time would write to the same files.

## 1. Settings

In [1]:
# ===== Edit these =====
DRIVE_ROOT    = "/content/drive/MyDrive/AML_Group_Project"   # team folder (a shortcut in your My Drive)
DATA_DIR      = f"{DRIVE_ROOT}/Dataset"                     # 3 CSVs + DPD_Small.7z
OUT_DIR       = f"{DRIVE_ROOT}/aml_runs"                    # checkpoints and results, shared with the team
REPO_URL     = "https://github.com/Daryl711/COS30028_ML_Project.git"
BRANCH        = "main"
FORCE_RETRAIN = False   # True = train again even if a finished run is already on Drive
# ======================

MODEL_DIR = "baseline/MultiTaskBaseline/DINOv2"
CONFIG    = "baseline/MultiTaskBaseline/DINOv2/config.yaml"
RUNS = {
    "multitask": "multitask_dinov2",
    "crop":      "indep_crop_dinov2",
    "disease":   "indep_disease_dinov2",
}
INDEP_EVAL = "indep_dinov2"   # evaluation folder for the crop + disease pair

## 2. Check the GPU and mount Google Drive

In [2]:
import os
import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then run this cell again."
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")

CSVS = ["DPD_Small_train.csv", "DPD_Small_test_seen.csv", "DPD_Small_test_unseen.csv"]
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"{DATA_DIR} not found. In Google Drive open 'Shared with me', right-click AML_Group_Project > "
        "Organize > Add shortcut > My Drive, then run this cell again. "
        "(If AML_Group_Project is a shared drive instead, set "
        "DRIVE_ROOT = '/content/drive/Shareddrives/AML_Group_Project'.)")
missing = [n for n in CSVS if not os.path.isfile(f"{DATA_DIR}/{n}")]
assert not missing, f"Missing from {DATA_DIR}: {missing}"
print("Dataset folder:", sorted(os.listdir(DATA_DIR)))

try:
    os.makedirs(OUT_DIR, exist_ok=True)
    with open(f"{OUT_DIR}/.write_test", "w") as f:
        f.write("ok")
    os.remove(f"{OUT_DIR}/.write_test")
except OSError as e:
    raise PermissionError(
        f"Can't write to {OUT_DIR} ({e}). You may only have Viewer access to the team folder: "
        "ask for Editor access, or set OUT_DIR = '/content/drive/MyDrive/aml_runs'.")
print("Results will be saved to", OUT_DIR)

AssertionError: No GPU. Runtime > Change runtime type > T4 GPU, then run this cell again.

## 3. Get the code

In [ ]:
import subprocess

REPO = "/content/proj"
if os.path.isdir(REPO):
    # Cloned earlier in this runtime. Restarting the session keeps /content, so pull to get the newest code.
    r = subprocess.run(["git", "-C", REPO, "pull", "-q", "--ff-only", "origin", BRANCH], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"git pull failed. Delete the old copy with !rm -rf {REPO} and run this cell again:\n" + r.stderr)
    print("Repo already cloned: pulled the latest code.")
else:
    url, token = REPO_URL, None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:
        pass  # no token: fine for a public repo
    r = subprocess.run(["git", "clone", "-q", "--branch", BRANCH, url, REPO], capture_output=True, text=True)
    if r.returncode:
        err = r.stderr.replace(token, "***") if token else r.stderr
        raise RuntimeError("git clone failed. If the repo is private, add a GITHUB_TOKEN in the Secrets panel:\n" + err)

%cd {REPO}
!git log -1 --format="Code version: %h %s (%cr)"
assert os.path.isdir(MODEL_DIR), f"{MODEL_DIR} isn't in the repo. Has it been committed and pushed?"
!pip install -q timm

## 4. Unpack the dataset to Colab's local disk

Reading thousands of small images from Drive is slow, so the CSVs and the image archive are copied into `dataset/` and unpacked locally (a few minutes).

In [ ]:
import glob
import shutil
import zipfile

IMG_DIR = "dataset/DPD_Small_dataset"
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
os.makedirs("dataset", exist_ok=True)
for name in CSVS:
    shutil.copy(f"{DATA_DIR}/{name}", f"dataset/{name}")

if os.path.isdir(IMG_DIR) and os.listdir(IMG_DIR):
    print("Images already unpacked.")
else:
    archive = f"{DATA_DIR}/DPD_Small.7z"
    if not os.path.isfile(archive):
        found = sorted(glob.glob(f"{DATA_DIR}/*.7z") + glob.glob(f"{DATA_DIR}/*.zip"))
        assert found, f"No .7z or .zip image archive in {DATA_DIR}"
        archive = found[0]
    local = "/content/" + os.path.basename(archive)
    print("Copying", archive)
    shutil.copy(archive, local)
    shutil.rmtree("/content/unpacked", ignore_errors=True)
    print("Unpacking...")
    if local.endswith(".zip"):
        with zipfile.ZipFile(local) as z:
            z.extractall("/content/unpacked")
    elif shutil.which("7z"):
        !7z x -y -o/content/unpacked "{local}" > /dev/null
    else:
        !pip -q install py7zr
        import py7zr
        with py7zr.SevenZipFile(local) as z:
            z.extractall("/content/unpacked")
    # Use the folder that holds the images, wherever it sits inside the archive.
    counts = {root: sum(f.lower().endswith(IMG_EXT) for f in fs) for root, _, fs in os.walk("/content/unpacked")}
    shutil.rmtree(IMG_DIR, ignore_errors=True)
    shutil.move(max(counts, key=counts.get), IMG_DIR)
    os.remove(local)

# Every image named in the CSVs must exist (first column; a header row, if any, is skipped).
import pandas as pd
names = pd.concat([pd.read_csv(f"dataset/{n}", header=None, usecols=[0])[0] for n in CSVS]).astype(str)
names = names[names.str.lower().str.endswith(IMG_EXT)]
have = set(os.listdir(IMG_DIR))
missing = [n for n in names if os.path.basename(n) not in have]
assert not missing, f"{len(missing)} images in the CSVs are missing, e.g. {missing[:3]}"
print(f"Images: {len(have)} (all {len(names)} CSV entries found)")

## 5. Save results to Drive

`outputs/` in the repo becomes a link to `OUT_DIR` (`AML_Group_Project/aml_runs`), so checkpoints and results are kept after the session ends.

In [ ]:
import csv
import yaml

if not os.path.islink("outputs"):
    shutil.rmtree("outputs", ignore_errors=True)
    os.symlink(OUT_DIR, "outputs")
print("outputs/ ->", os.readlink("outputs"))

with open(CONFIG) as f:
    CFG = yaml.safe_load(f)
EPOCHS, PATIENCE = CFG["train"]["epochs"], CFG["train"].get("patience", 0)


def run_status(exp):
    """'done', 'partial' or 'missing' for a training run saved on Drive."""
    hist = f"outputs/{exp}/history.csv"
    if not (os.path.isfile(hist) and os.path.isfile(f"outputs/{exp}/best.pt")):
        return "missing"
    with open(hist) as f:
        scores = [float(r["val_score"]) for r in csv.DictReader(f)]
    if not scores:
        return "partial"
    best_epoch = scores.index(max(scores)) + 1          # same rule as train.py
    early_stop = PATIENCE and len(scores) - best_epoch >= PATIENCE
    return "done" if len(scores) >= EPOCHS or early_stop else "partial"


def shell(cmd):
    print("$", cmd)
    get_ipython().system(cmd)
    if get_ipython().user_ns.get("_exit_code", 0):
        raise RuntimeError("Command failed, see the error above.")


def train(mode):
    exp = RUNS[mode]
    status = run_status(exp)
    if status == "done" and not FORCE_RETRAIN:
        print(f"{exp}: already finished on Drive, skipping. Set FORCE_RETRAIN = True to train it again.")
        return
    if status == "partial":
        print(f"{exp}: found an unfinished run. Training restarts from epoch 1.")
    shell(f'python "{MODEL_DIR}/train.py" --config "{CONFIG}" --mode {mode} --experiment {exp}')


def evaluate(modes, out_name):
    exps = [RUNS[m] for m in modes]
    not_done = [e for e in exps if run_status(e) != "done"]
    if not_done:
        print(f"Skipping {out_name}: training not finished for {not_done}")
        return
    ckpts = [f"outputs/{e}/best.pt" for e in exps]
    metrics = f"outputs/{out_name}/eval/metrics.json"
    if (os.path.isfile(metrics) and not FORCE_RETRAIN
            and os.path.getmtime(metrics) > max(os.path.getmtime(c) for c in ckpts)):
        print(f"{out_name}: already evaluated, skipping.")
        return
    shell(f'python "{MODEL_DIR}/evaluate.py" --checkpoints {" ".join(ckpts)} --out outputs/{out_name}/eval')


for exp in RUNS.values():
    print(f"{exp:28s} {run_status(exp)}")

## 6. Train the multi-task model

In [ ]:
train("multitask")

## 7. Train the independent crop model

In [ ]:
train("crop")

## 8. Train the independent disease model

In [ ]:
train("disease")

## 9. Evaluate on test_seen and test_unseen

In [ ]:
evaluate(["multitask"], RUNS["multitask"])
evaluate(["crop", "disease"], INDEP_EVAL)

## 10. Results

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

rows = {}
for name in [RUNS["multitask"], INDEP_EVAL]:
    path = f"outputs/{name}/eval/metrics.json"
    if not os.path.isfile(path):
        continue
    with open(path) as f:
        m = json.load(f)
    rows[name] = {
        "seen crop": m["seen"]["crop"]["top1"],
        "seen disease": m["seen"]["disease"]["top1"],
        "seen pair": m["seen"]["composition"]["top1"],
        "unseen crop": m["unseen"]["crop"]["top1"],
        "unseen disease": m["unseen"]["disease"]["top1"],
        "unseen pair": m["unseen"]["composition"]["top1"],
        "HM": m["harmonic_mean"]["top1"],
        "best HM (calibrated)": m["calibrated"]["best_hm"],
        "AUC": m["calibrated"]["auc"],
    }
if rows:
    print("Top-1 accuracy")
    display(pd.DataFrame(rows).T.round(4))
else:
    print("No evaluation results yet.")

# Training curves
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for exp in RUNS.values():
    hist = f"outputs/{exp}/history.csv"
    if os.path.isfile(hist):
        h = pd.read_csv(hist)
        ax[0].plot(h["epoch"], h["val_score"], label=exp)
        ax[1].plot(h["epoch"], h["val_loss"], label=exp)
ax[0].set(title="Validation score (mean top-1 over heads)", xlabel="epoch")
ax[1].set(title="Validation loss", xlabel="epoch")
for a in ax:
    a.legend()
plt.show()

for name in [RUNS["multitask"], INDEP_EVAL]:
    for png in ["calibration_curve.png", "top_confusions_seen.png"]:
        path = f"outputs/{name}/eval/{png}"
        if os.path.isfile(path):
            print(path)
            display(Image(path, width=600))

## 11. Download the results to commit them

Zips the small result files (no checkpoints) for this model. Unzip inside the repo's `outputs/` folder on your computer, then commit as usual. The checkpoints (`*.pt`) stay on Drive.

In [ ]:
from google.colab import files

names = list(RUNS.values()) + [INDEP_EVAL]
zip_path = f"/content/results_{RUNS['multitask']}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in names:
        for root, _, fs in os.walk(f"outputs/{name}"):
            for fn in fs:
                if not fn.endswith(".pt"):
                    p = os.path.join(root, fn)
                    z.write(p, os.path.relpath(p, "outputs"))
print("Files:", len(zipfile.ZipFile(zip_path).namelist()))
files.download(zip_path)